# AlphaGenome CRE Experiments

A unified notebook for running **systematic CRE (cis-regulatory element) perturbation experiments** with the [AlphaGenome API](https://github.com/google-deepmind/alphagenome).

**Quick start:** Fill in section 1 (gene, cell type, replicates), then jump to any test section — each one is self-contained.

## What are CREs?

Cis-regulatory elements — enhancers, promoters, silencers — are stretches of non-coding DNA that act as molecular switches, controlling *when* and *where* genes are turned on or off. Understanding which CREs regulate a given gene is fundamental to interpreting non-coding genetic variation and disease mechanisms.

## What is *in silico* mutagenesis (ISM)?

Instead of performing wet-lab experiments, we computationally edit DNA sequences and use a trained model (AlphaGenome) to predict the effect on gene expression. Think of it as a virtual experiment: we can systematically perturb every region around a gene and measure which perturbations matter — all without touching a pipette.

## Why would I use this?

- **Understand which enhancers regulate your favourite gene** in a specific cell type or cancer cell line.
- **Interpret GWAS hits in non-coding regions** — identify which regulatory elements a variant disrupts.
- **Map the regulatory landscape** around a disease-associated gene to prioritize functional follow-up.

## What this notebook does

We implement four [CREME](https://www.nature.com/articles/s41588-024-01923-3)-style experiments (Toneyan and Koo, Nature Genetics 2024):

1. **Necessity test** — Tile a locus, shuffle each tile, identify essential CREs.
2. **Sufficiency test** — Plant each CRE into a fully-shuffled background to see if it can drive expression on its own.
3. **Higher-order interaction** — Greedy ablation to discover CRE cooperativity.
4. **CRISPRi tiling scan** — Fine-resolution shuffle scan with full track outputs.

Each test section ships with a short **mock animation** explaining the concept, and a **real-data playback** that reveals the result tile-by-tile on a [CoolBox](https://github.com/GangCaoLab/CoolBox)-rendered genome browser.

### How deletion is simulated

We cannot truly "delete" DNA *in silico* — removing bases would change the sequence length and shift the reading frame of the model. Instead, we simulate deletion using **dinucleotide shuffling**: the reference sequence of each tile is replaced with a randomly permuted version that preserves the exact frequencies of all 16 dinucleotides (AA, AC, AG, AT, ...). This destroys transcription factor binding sites and regulatory grammar while keeping GC content and local sequence composition intact, providing a biologically meaningful null. Each tile is shuffled multiple times and scored in both forward and reverse-complement orientations; the effect is averaged across all replicates to produce a robust estimate. This is the same perturbation strategy used by [CREME](https://www.nature.com/articles/s41588-024-01923-3).

## Setup

### AlphaGenome API key (required)

One-time setup:
1. **Get an API key**: visit the [AlphaGenome API page](https://aistudio.google.com/apikey) and create a new key.
2. **Add it to Colab Secrets**: click the **key icon** in the left sidebar, then **+ Add new secret**. Set the name to `ALPHAGENOME_API_KEY` and paste your key as the value. Toggle **Notebook access** on.
3. The code cell below reads the key via `google.colab.userdata.get('ALPHAGENOME_API_KEY')`.

In [ ]:
#@title Install & import dependencies { display-mode: "form" }
import os
from IPython.display import clear_output

if not os.path.exists('ALPHAGENOME_READY'):
    !pip install alphagenome pyBigWig
    open('ALPHAGENOME_READY', 'w').close()
    clear_output()
    print('Installation complete.')
else:
    print('Already installed, skipping.')

import dataclasses, json, re, requests, tempfile, zipfile
from pathlib import Path
from typing import Any
from google.colab import userdata
from alphagenome.data import gene_annotation, genome, transcript as transcript_utils
from alphagenome.models import dna_client, variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Sharper plots everywhere in the notebook
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.linewidth'] = 0.8

try:
    import pyBigWig
    _HAS_BIGWIG = True
except Exception as _e:
    _HAS_BIGWIG = False
    print(f'pyBigWig unavailable ({_e!s}) — BigWig export will fall back to BedGraph.')

print('Ready.')

In [ ]:
#@title Setup: AlphaGenome initialization { display-mode: "form" }

def init(api_key, seq_length=dna_client.SEQUENCE_LENGTH_1MB):
    dna_model = dna_client.create(api_key)
    gtf = pd.read_feather('https://storage.googleapis.com/alphagenome/reference/gencode/hg38/gencode.v46.annotation.gtf.gz.feather')
    gtf_tx = gene_annotation.filter_protein_coding(gtf)
    gtf_tx = gene_annotation.filter_to_mane_select_transcript(gtf_tx)
    transcript_extractor = transcript_utils.TranscriptExtractor(gtf_tx)
    print(f'Model connected. {len(gtf)} GTF annotations loaded.')
    return dna_model, gtf, transcript_extractor

In [ ]:
#@title CREME experiment framework { display-mode: "form" }

_SEQ_CACHE = {}

def dinuc_shuffle(seq, num_shufs=None, rng=None):
    if rng is None: rng = np.random.RandomState()
    arr = np.frombuffer(bytearray(seq, 'utf8'), dtype=np.int8)
    chars, tokens = np.unique(arr, return_inverse=True)
    shuf_next_inds = []

    for t in range(len(chars)):
        mask = tokens[:-1] == t; inds = np.where(mask)[0]; shuf_next_inds.append(inds + 1)
    results = []

    for _ in range(num_shufs if num_shufs else 1):
        for t in range(len(chars)):
            inds = np.arange(len(shuf_next_inds[t]))
            if len(inds) > 1: inds[:-1] = rng.permutation(len(inds) - 1)
            shuf_next_inds[t] = shuf_next_inds[t][inds]
        counters = [0] * len(chars); ind = 0; result = np.empty_like(tokens); result[0] = tokens[ind]
        for j in range(1, len(tokens)):
            t = tokens[ind]; ind = shuf_next_inds[t][counters[t]]; counters[t] += 1; result[j] = tokens[ind]
        results.append(chars[result].tobytes().decode('ascii'))

    return results if num_shufs else results[0]

def _fetch_ref_seq(interval):
    key = f'{interval.chromosome}:{interval.start}-{interval.end}'
    if key in _SEQ_CACHE: 
        return _SEQ_CACHE[key]

    url = f'https://api.genome.ucsc.edu/getData/sequence?genome=hg38&chrom={interval.chromosome}&start={interval.start}&end={interval.end}'
    resp = requests.get(url, timeout=30); resp.raise_for_status()
    seq = resp.json()['dna'].upper(); _SEQ_CACHE[key] = seq; return seq

def _reverse_complement(seq):
    return seq.translate(str.maketrans('ACGT', 'TGCA'))[::-1]

def _make_shuffle_variant(tile, ref_seq, shuf_seq, name=None):
    return genome.Variant(chromosome=tile.chromosome, position=tile.start + 1, reference_bases=ref_seq, alternate_bases=shuf_seq, name=name or f'shuf_{tile.chromosome}:{tile.start}-{tile.end}')

def _average_shuffle_scores(dfs):
    combined = pd.concat(dfs, ignore_index=True)
    drop_cols = {'raw_score', 'quantile_score', 'variant_id', 'scored_interval'}

    for c in combined.columns:
        if c in drop_cols: continue
        try: combined[c].unique()
        except TypeError: drop_cols.add(c)
    group_cols = [c for c in combined.columns if c not in drop_cols]

    return combined.groupby(group_cols, as_index=False, sort=False, dropna=False).agg(raw_score=('raw_score', 'mean'), quantile_score=('quantile_score', 'mean'))

# --- Result dataclasses ---

@dataclasses.dataclass
class NecessityResult:
    tile_effects: pd.DataFrame; scores: pd.DataFrame; target_gene: str
    context_chrom: str; context_start: int; context_end: int; ontology_curie: str | None = None

    def to_csv(self, path, include_full_scores=False):
        self.tile_effects.to_csv(f'{path}_data.csv', index=False)
        meta = {'target_gene': self.target_gene, 'context_chrom': self.context_chrom, 'context_start': self.context_start, 'context_end': self.context_end, 'result_type': 'NecessityResult', 'ontology_curie': self.ontology_curie}
        with open(f'{path}_meta.json', 'w') as f: json.dump(meta, f, indent=2)
            
        if include_full_scores: self.scores.to_csv(f'{path}_full_scores.csv', index=False)

    @classmethod
    def from_csv(cls, path):
        tile_effects = pd.read_csv(f'{path}_data.csv')
        with open(f'{path}_meta.json') as f: meta = json.load(f)
        fp = Path(f'{path}_full_scores.csv'); scores = pd.read_csv(fp) if fp.exists() else pd.DataFrame()

        return cls(tile_effects=tile_effects, scores=scores, target_gene=meta['target_gene'], context_chrom=meta['context_chrom'], context_start=meta['context_start'], context_end=meta['context_end'], ontology_curie=meta.get('ontology_curie'))

@dataclasses.dataclass
class InteractionResult:
    rounds: pd.DataFrame; target_gene: str; ontology_curie: str

    def to_csv(self, path):
        self.rounds.to_csv(f'{path}_data.csv', index=False)
        meta = {'target_gene': self.target_gene, 'ontology_curie': self.ontology_curie, 'result_type': 'InteractionResult'}
        with open(f'{path}_meta.json', 'w') as f: json.dump(meta, f, indent=2)

    @classmethod
    def from_csv(cls, path):
        rounds = pd.read_csv(f'{path}_data.csv')
        with open(f'{path}_meta.json') as f: meta = json.load(f)

        return cls(rounds=rounds, target_gene=meta['target_gene'], ontology_curie=meta['ontology_curie'])

@dataclasses.dataclass
class CRISPRiResult:
    tile_scores: pd.DataFrame; tile_outputs: list; target_gene: str
    context_chrom: str; context_start: int; context_end: int

    def to_csv(self, path):
        self.tile_scores.to_csv(f'{path}_data.csv', index=False)
        meta = {'target_gene': self.target_gene, 'context_chrom': self.context_chrom, 'context_start': self.context_start, 'context_end': self.context_end, 'result_type': 'CRISPRiResult'}
        with open(f'{path}_meta.json', 'w') as f: json.dump(meta, f, indent=2)

    @classmethod
    def from_csv(cls, path):
        tile_scores = pd.read_csv(f'{path}_data.csv')
        with open(f'{path}_meta.json') as f: meta = json.load(f)

        return cls(tile_scores=tile_scores, tile_outputs=None, target_gene=meta['target_gene'], context_chrom=meta['context_chrom'], context_start=meta['context_start'], context_end=meta['context_end'])

# --- Cell type search ---

def search_cell_types(model, query, _cache={}):
    if 'all_biosamples' not in _cache:
        dummy_iv = genome.Interval('chr1', 0, dna_client.SEQUENCE_LENGTH_16KB)
        dummy_var = genome.Variant(chromosome='chr1', position=100, reference_bases='A', alternate_bases='T', name='dummy')
        result = model.score_variant(interval=dummy_iv, variant=dummy_var, variant_scorers=[variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']])
        df = variant_scorers.tidy_scores([result], match_gene_strand=True)
        bio_cols = [c for c in df.columns if c.startswith(('ontology_curie', 'biosample'))] or ['ontology_curie']
        _cache['all_biosamples'] = df[bio_cols].drop_duplicates().reset_index(drop=True)

    biosamples = _cache['all_biosamples']; q = query.lower()
    mask = biosamples.apply(lambda row: any(q in str(v).lower() for v in row), axis=1)

    return biosamples[mask].sort_values(biosamples.columns[0]).reset_index(drop=True)

# --- Main experiment class ---

class CREExperiment:
    def __init__(self, model, gene_symbol, gtf, transcript_extractor, ontology_terms, seq_length=dna_client.SEQUENCE_LENGTH_1MB):
        self.model, self.gene_symbol, self.gtf = model, gene_symbol, gtf
        self.transcript_extractor, self.ontology_terms, self.seq_length = transcript_extractor, ontology_terms, seq_length
        self.gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
        self.context_interval = self.gene_interval.resize(seq_length)

    @staticmethod
    def tile_region(region, tile_width=5000, step=None):
        if step is None: step = tile_width
        tiles = []; pos = region.start

        while pos < region.end:
            tiles.append(genome.Interval(region.chromosome, pos, min(pos + tile_width, region.end))); pos += step

        return tiles

    def necessity_test(self, tiles, scorers=None, target_gene=None, n_shuffles=10):
        target_gene = target_gene or self.gene_symbol
        if scorers is None: scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]
        rng = np.random.RandomState(42); per_tile_dfs = []

        for i, tile in enumerate(tqdm(tiles, desc='Necessity test')):
            ref_seq = _fetch_ref_seq(tile); shuffled_seqs = dinuc_shuffle(ref_seq, num_shufs=n_shuffles, rng=rng)
            replicate_dfs = []
            for s, shuf_seq in enumerate(shuffled_seqs):
                for orientation, seq in [('fwd', shuf_seq), ('rc', _reverse_complement(shuf_seq))]:
                    var = _make_shuffle_variant(tile, ref_seq, seq, name=f'Tile_{i+1}_shuf{s+1}_{orientation}')
                    res = self.model.score_variant(interval=self.context_interval, variant=var, variant_scorers=scorers)
                    replicate_dfs.append(variant_scorers.tidy_scores([res], match_gene_strand=True))
            avg_df = _average_shuffle_scores(replicate_dfs)
            avg_df['tile_idx'] = i; avg_df['tile_label'] = f'Tile {i+1}'
            avg_df['chrom'] = tile.chromosome; avg_df['tile_start'] = tile.start; avg_df['tile_end'] = tile.end
            per_tile_dfs.append(avg_df)

        scores_df = pd.concat(per_tile_dfs, ignore_index=True)
        gene_scores = scores_df[scores_df['gene_name'] == target_gene].copy()

        return NecessityResult(tile_effects=gene_scores, scores=scores_df, target_gene=target_gene, context_chrom=self.context_interval.chromosome, context_start=self.context_interval.start, context_end=self.context_interval.end)

    def interaction_test(self, tiles, scorers=None, target_gene=None, num_rounds=None, ontology_curie=None, n_shuffles=10):
        target_gene = target_gene or self.gene_symbol
        if scorers is None: scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]
        if num_rounds is None: num_rounds = len(tiles)
        num_rounds = min(num_rounds, len(tiles))
        remaining = list(range(len(tiles))); records = []; cumulative = 0.0

        for round_num in range(1, num_rounds + 1):
            nec = self.necessity_test([tiles[idx] for idx in remaining], scorers=scorers, target_gene=target_gene, n_shuffles=n_shuffles)
            effects = nec.tile_effects
            if ontology_curie is not None: 
                effects = effects[effects['ontology_curie'] == ontology_curie]
            elif len(effects) > 0: 
                ontology_curie = effects['ontology_curie'].iloc[0]
            if len(effects) == 0: 
                break
            worst_idx = effects['raw_score'].abs().idxmax(); worst_row = effects.loc[worst_idx]
            tile_num = int(worst_row['tile_label'].split()[-1]) - 1; original_idx = remaining[tile_num]; tile = tiles[original_idx]
            cumulative += worst_row['raw_score']
            records.append({'round': round_num, 'tile_removed': f'Tile {original_idx + 1}', 'tile_chrom': tile.chromosome, 'tile_start': tile.start, 'tile_end': tile.end, 'delta': float(worst_row['raw_score']), 'cumulative_effect': float(cumulative)})
            remaining.pop(tile_num)
            if not remaining: 
                break

        return InteractionResult(rounds=pd.DataFrame(records), target_gene=target_gene, ontology_curie=ontology_curie or '')

    def crispri_scan(self, tiles, outputs=None, target_gene=None, n_shuffles=10):
        target_gene = target_gene or self.gene_symbol
        if outputs is None: 
            outputs = [dna_client.OutputType.RNA_SEQ]
        rng = np.random.RandomState(42); tile_outputs = []; per_tile_dfs = []
        scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]

        for i, tile in enumerate(tqdm(tiles, desc='CRISPRi scan')):
            ref_seq = _fetch_ref_seq(tile); shuffled_seqs = dinuc_shuffle(ref_seq, num_shufs=n_shuffles, rng=rng)
            first_var = _make_shuffle_variant(tile, ref_seq, shuffled_seqs[0], name=f'CRISPRi_Tile_{i+1}_shuf1_fwd')
            voutput = self.model.predict_variant(interval=self.context_interval, variant=first_var, requested_outputs=outputs, ontology_terms=self.ontology_terms)
            tile_outputs.append((tile, first_var, voutput))
            replicate_dfs = []
            for s, shuf_seq in enumerate(shuffled_seqs):
                for orientation, seq in [('fwd', shuf_seq), ('rc', _reverse_complement(shuf_seq))]:
                    var = _make_shuffle_variant(tile, ref_seq, seq, name=f'CRISPRi_Tile_{i+1}_shuf{s+1}_{orientation}')
                    res = self.model.score_variant(interval=self.context_interval, variant=var, variant_scorers=scorers)
                    replicate_dfs.append(variant_scorers.tidy_scores([res], match_gene_strand=True))
            avg_df = _average_shuffle_scores(replicate_dfs)
            avg_df['tile_idx'] = i; avg_df['tile_label'] = f'Tile {i+1}'
            avg_df['chrom'] = tile.chromosome; avg_df['tile_start'] = tile.start; avg_df['tile_end'] = tile.end
            per_tile_dfs.append(avg_df)

        tile_scores = pd.concat(per_tile_dfs, ignore_index=True)
        
        return CRISPRiResult(tile_scores=tile_scores, tile_outputs=tile_outputs, target_gene=target_gene, context_chrom=self.context_interval.chromosome, context_start=self.context_interval.start, context_end=self.context_interval.end)

# --- Sufficiency test (a CRE alone in a fully-shuffled context) ---

@dataclasses.dataclass
class SufficiencyResult:
    tile_effects: pd.DataFrame; scores: pd.DataFrame; target_gene: str
    context_chrom: str; context_start: int; context_end: int; ontology_curie: str | None = None

    def to_csv(self, path, include_full_scores=False):
        self.tile_effects.to_csv(f'{path}_data.csv', index=False)
        meta = {'target_gene': self.target_gene, 'context_chrom': self.context_chrom, 'context_start': self.context_start, 'context_end': self.context_end, 'result_type': 'SufficiencyResult', 'ontology_curie': self.ontology_curie}
        with open(f'{path}_meta.json', 'w') as f: json.dump(meta, f, indent=2)
        if include_full_scores: self.scores.to_csv(f'{path}_full_scores.csv', index=False)

    @classmethod
    def from_csv(cls, path):
        tile_effects = pd.read_csv(f'{path}_data.csv')
        with open(f'{path}_meta.json') as f: meta = json.load(f)
        fp = Path(f'{path}_full_scores.csv'); scores = pd.read_csv(fp) if fp.exists() else pd.DataFrame()
        return cls(tile_effects=tile_effects, scores=scores, target_gene=meta['target_gene'], context_chrom=meta['context_chrom'], context_start=meta['context_start'], context_end=meta['context_end'], ontology_curie=meta.get('ontology_curie'))


def _sufficiency_test_impl(self, tiles, scorers=None, target_gene=None, n_shuffles=2):
    """For each tile, plant its original sequence into an otherwise dinuc-shuffled
    1 MB context, score gene expression. LFC(ALT vs REF) measures how much the CRE
    alone can drive expression above the silent (shuffled-everywhere) background.

    Implementation note: AlphaGenome's score_variant requires the variant to be
    *strictly* inside the evaluation interval. A variant whose span equals the
    interval is rejected with INVALID_ARGUMENT. We trim 1 base from each end of
    the synthetic ref/alt sequences and offset the position accordingly. The
    1-base edges of a 1 MB context are far outside the gene's effective receptive
    field, so this is biologically inconsequential.
    """
    target_gene = target_gene or self.gene_symbol
    if scorers is None:
        scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]
    rng = np.random.RandomState(42)
    ctx_seq = _fetch_ref_seq(self.context_interval)
    per_tile_dfs = []

    TRIM = 1  # bases removed from each end so variant span < interval span

    for i, tile in enumerate(tqdm(tiles, desc='Sufficiency test')):
        tile_offset = tile.start - self.context_interval.start
        tile_len = tile.end - tile.start
        original_tile = ctx_seq[tile_offset:tile_offset + tile_len]
        replicate_dfs = []
        for s in range(n_shuffles):
            bg = dinuc_shuffle(ctx_seq, num_shufs=None, rng=rng)
            bg_with_cre = bg[:tile_offset] + original_tile + bg[tile_offset + tile_len:]
            ref_seq = bg[TRIM:len(bg) - TRIM]
            alt_seq = bg_with_cre[TRIM:len(bg_with_cre) - TRIM]
            for orientation, (rs, as_) in [('fwd', (ref_seq, alt_seq)), ('rc', (_reverse_complement(ref_seq), _reverse_complement(alt_seq)))]:
                var = genome.Variant(chromosome=tile.chromosome, position=self.context_interval.start + TRIM + 1, reference_bases=rs, alternate_bases=as_, name=f'Suff_Tile_{i+1}_shuf{s+1}_{orientation}')
                res = self.model.score_variant(interval=self.context_interval, variant=var, variant_scorers=scorers)
                replicate_dfs.append(variant_scorers.tidy_scores([res], match_gene_strand=True))
        avg_df = _average_shuffle_scores(replicate_dfs)
        avg_df['tile_idx'] = i; avg_df['tile_label'] = f'Tile {i+1}'
        avg_df['chrom'] = tile.chromosome; avg_df['tile_start'] = tile.start; avg_df['tile_end'] = tile.end
        per_tile_dfs.append(avg_df)

    scores_df = pd.concat(per_tile_dfs, ignore_index=True)
    gene_scores = scores_df[scores_df['gene_name'] == target_gene].copy()
    return SufficiencyResult(tile_effects=gene_scores, scores=scores_df, target_gene=target_gene, context_chrom=self.context_interval.chromosome, context_start=self.context_interval.start, context_end=self.context_interval.end)


CREExperiment.sufficiency_test = _sufficiency_test_impl


In [ ]:
#@title Plotting functions { display-mode: "form" }

def _format_pos(pos): return f'{pos:.3e}'

def _get_tile_scores(result, ontology_curie=None):
    df = result.tile_effects.copy()
    if ontology_curie is not None: 
        df = df[df['ontology_curie'] == ontology_curie]
    else: curie = df['ontology_curie'].iloc[0]; df = df[df['ontology_curie'] == curie]; ontology_curie = curie

    if len(df) > df['tile_idx'].nunique():
        keep = [c for c in ['tile_idx', 'tile_label', 'chrom', 'tile_start', 'tile_end'] if c in df.columns]
        df = df.groupby(keep, as_index=False, sort=False).agg(**{k: (k, 'mean') for k in ['raw_score', 'quantile_score'] if k in df.columns})
    return df.sort_values('tile_idx'), ontology_curie

def _bar_chart(df, chrom, title, ylabel='LFC (log2 ALT/REF)', score_col='raw_score', ax=None):
    if ax is None: _, ax = plt.subplots(figsize=(14, 4))

    for _, row in df.iterrows():
        ts, te = row['tile_start'], row['tile_end']; score = row[score_col]
        ax.bar((ts + te) / 2, score, width=(te - ts) * 0.9, color='#2166ac' if score < 0 else '#b2182b', edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8); ax.set_ylabel(ylabel); ax.set_xlabel(f'{chrom} position'); ax.set_title(title)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: _format_pos(int(x)))); ax.spines[['top', 'right']].set_visible(False); plt.tight_layout()

    return ax

def plot_necessity_bar(result, ontology_curie=None, ax=None):
    df, ontology_curie = _get_tile_scores(result, ontology_curie)

    return _bar_chart(df, df['chrom'].iloc[0], f'Necessity test: {result.target_gene} | {ontology_curie}', ax=ax)

def plot_necessity_heatmap(result, gtf=None, extra_curies=(), ax=None):
    df = result.tile_effects.copy().sort_values('tile_idx')
    primary_curie = getattr(result, 'ontology_curie', None)

    if extra_curies is not None:
        keep = set(extra_curies); (keep.add(primary_curie) if primary_curie else None)
        if keep: 
            df = df[df['ontology_curie'].isin(keep)]

    tiles_df = df[['tile_idx', 'chrom', 'tile_start', 'tile_end']].drop_duplicates().sort_values('tile_idx')
    num_tiles = len(tiles_df); tile_starts, tile_ends = tiles_df['tile_start'].values, tiles_df['tile_end'].values
    tile_widths = tile_ends - tile_starts; chrom = tiles_df['chrom'].iloc[0]
    agg_df = df.groupby(['ontology_curie', 'tile_idx'], as_index=False, sort=False).agg(raw_score=('raw_score', 'mean'))
    curies = sorted(agg_df['ontology_curie'].unique())
    if primary_curie and primary_curie in curies: 
        curies.remove(primary_curie); curies.insert(0, primary_curie)
    matrix = np.full((len(curies), num_tiles), np.nan)

    for _, row in agg_df.iterrows(): matrix[curies.index(row['ontology_curie']), int(row['tile_idx'])] = row['raw_score']
    region_start, region_end, tw = tile_starts.min(), tile_ends.max(), tile_widths[0]
    has_genes = gtf is not None

    if has_genes:
        fig, (ax_gene, ax_heat) = plt.subplots(2, 1, figsize=(14, max(3, 0.5 * len(curies) + 2)), height_ratios=[1, max(2, len(curies) * 0.6)], sharex=True)
    else:
        fig, ax_heat = plt.subplots(figsize=(14, max(3, 0.5 * len(curies) + 1)))
    vmax = np.nanmax(np.abs(matrix)) if not np.all(np.isnan(matrix)) else 1e-6

    for ci in range(len(curies)):
        for ti in range(num_tiles):
            val = matrix[ci, ti]
            if np.isnan(val): continue
            rect = mpatches.FancyBboxPatch((tile_starts[ti], ci - 0.4), tile_widths[ti], 0.8, boxstyle='round,pad=0', facecolor=plt.cm.RdBu_r((val / vmax + 1) / 2), edgecolor='white', linewidth=0.5)
            ax_heat.add_patch(rect)

    ax_heat.set_xlim(region_start - tw * 0.1, region_end + tw * 0.1); ax_heat.set_ylim(-0.5, len(curies) - 0.5)
    ax_heat.set_yticks(range(len(curies))); ax_heat.set_yticklabels(curies, fontsize=8)
    ax_heat.set_xlabel(f'{chrom} position'); ax_heat.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: _format_pos(int(x))))
    ax_heat.spines[['top', 'right']].set_visible(False)
    norm = mcolors.Normalize(vmin=-vmax, vmax=vmax); sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=norm); sm.set_array([])
    plt.colorbar(sm, ax=ax_heat, label='LFC', shrink=0.6, pad=0.02)

    if has_genes:
        pad = int((region_end - region_start) * 0.05); view_start, view_end = region_start - pad, region_end + pad
        genes_in_view = gtf[(gtf['Chromosome'] == chrom) & (gtf['End'] >= view_start) & (gtf['Start'] <= view_end)].copy()
        if 'Feature' in genes_in_view.columns: genes_in_view = genes_in_view[genes_in_view['Feature'] == 'gene']
        if 'gene_type' in genes_in_view.columns:
            pc = genes_in_view[genes_in_view['gene_type'] == 'protein_coding']
            if len(pc) > 0: genes_in_view = pc
        if 'gene_name' in genes_in_view.columns: 
            genes_in_view = genes_in_view.drop_duplicates(subset='gene_name')

        ax_gene.set_xlim(region_start - tw * 0.1, region_end + tw * 0.1); ax_gene.set_ylim(-0.5, max(1, len(genes_in_view)) - 0.5)
        ax_gene.set_yticks([]); ax_gene.set_title(f'Necessity: {result.target_gene} | {chrom}:{_format_pos(region_start)}-{_format_pos(region_end)}')
        ax_gene.spines[['top', 'right', 'bottom', 'left']].set_visible(False); ax_gene.tick_params(bottom=False)

        for yi, (_, gene) in enumerate(genes_in_view.iterrows()):
            g_start, g_end, strand = gene['Start'], gene['End'], gene.get('Strand', '+'); name = gene.get('gene_name', '')
            ax_gene.plot([g_start, g_end], [yi, yi], color='#333333', linewidth=2, solid_capstyle='butt')
            tss = g_start if strand == '+' else g_end; dx = (region_end - region_start) * 0.015 * (-1 if strand == '-' else 1)
            ax_gene.annotate('', xy=(tss + dx, yi), xytext=(tss, yi), arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))
            ax_gene.text((g_start + g_end) / 2, yi + 0.25, name, ha='center', va='bottom', fontsize=8, fontstyle='italic')

    plt.tight_layout(); return fig

def plot_necessity_genome(result, transcript_extractor=None, ontology_curie=None):
    df, ontology_curie = _get_tile_scores(result, ontology_curie)
    chrom = df['chrom'].iloc[0]; region_start, region_end = df['tile_start'].min(), df['tile_end'].max()
    pad = int((region_end - region_start) * 0.1); view_iv = genome.Interval(chrom, region_start - pad, region_end + pad)
    has_tx = transcript_extractor is not None

    if has_tx:
        fig, (ax_gene, ax_tiles) = plt.subplots(2, 1, figsize=(14, 4), height_ratios=[1, 2], sharex=True)
        transcripts = transcript_extractor.extract(view_iv); annot = plot_components.TranscriptAnnotation(transcripts)
        annot.plot_ax(ax_gene, axis_index=0, interval=view_iv); ax_gene.set_title(f'Necessity: {result.target_gene} tiles')
        ax_gene.spines[['top', 'right', 'bottom']].set_visible(False); ax_gene.tick_params(bottom=False)
    else: 
        fig, ax_tiles = plt.subplots(figsize=(14, 2))

    scores, tile_starts, tile_ends = df['raw_score'].values, df['tile_start'].values, df['tile_end'].values
    vmax = max(abs(scores.min()), abs(scores.max())) if len(scores) else 1e-6
    norm = mcolors.Normalize(vmin=-vmax, vmax=vmax); cmap = plt.cm.RdBu_r

    for i in range(len(df)):
        ts, te, tw = tile_starts[i], tile_ends[i], tile_ends[i] - tile_starts[i]
        rect = mpatches.FancyBboxPatch((ts, 0.1), tw, 0.8, boxstyle='round,pad=0', facecolor=cmap(norm(scores[i])), edgecolor='black', linewidth=0.5)
        ax_tiles.add_patch(rect); ax_tiles.text(ts + tw / 2, 0.5, df['tile_label'].values[i], ha='center', va='center', fontsize=7)
    ax_tiles.set_xlim(view_iv.start, view_iv.end); ax_tiles.set_ylim(0, 1); ax_tiles.set_xlabel(f'{chrom} position'); ax_tiles.set_yticks([])
    ax_tiles.set_title(f'Tile necessity scores ({ontology_curie})')
    ax_tiles.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: _format_pos(int(x)))); ax_tiles.spines[['top', 'right', 'left']].set_visible(False)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([]); plt.colorbar(sm, ax=ax_tiles, label='LFC', shrink=0.6, pad=0.02)
    plt.tight_layout(); return fig

def plot_interaction_waterfall(result, ax=None):
    if ax is None: _, ax = plt.subplots(figsize=(8, 5))
    rdf = result.rounds if isinstance(result.rounds, pd.DataFrame) else pd.DataFrame(result.rounds)
    rounds_num = [0] + rdf['round'].tolist(); cum = [0.0] + rdf['cumulative_effect'].tolist()
    ax.step(rounds_num, cum, where='post', color='#2166ac', linewidth=2, marker='o', markersize=6)
    ax.fill_between(rounds_num, cum, step='post', alpha=0.15, color='#2166ac')

    for _, r in rdf.iterrows():
        ax.annotate(r['tile_removed'], xy=(r['round'], r['cumulative_effect']), xytext=(5, 10), textcoords='offset points', fontsize=8, ha='left', arrowprops=dict(arrowstyle='->', color='grey', lw=0.8))

    ax.set_xlabel('Tiles removed'); ax.set_ylabel('Cumulative LFC from WT')
    ax.set_title(f'Interaction test: {result.target_gene} | {result.ontology_curie}')
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--'); ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); return ax

def plot_interaction_contribution(result, ax=None):
    rdf = result.rounds if isinstance(result.rounds, pd.DataFrame) else pd.DataFrame(result.rounds)

    return _bar_chart(rdf, rdf['tile_chrom'].iloc[0], f'Per-round contribution: {result.target_gene} | {result.ontology_curie}', ylabel='Delta LFC', score_col='delta', ax=ax)

def plot_crispri_summary(result, ontology_curie=None, ax=None):
    df = result.tile_scores.copy(); df = df[df['gene_name'] == result.target_gene]
    if ontology_curie is not None: 
        df = df[df['ontology_curie'] == ontology_curie]
    elif 'ontology_curie' in df.columns and len(df) > 0: 
        curie = df['ontology_curie'].iloc[0]; df = df[df['ontology_curie'] == curie]; ontology_curie = curie
    keep = [c for c in ['tile_idx', 'tile_label', 'chrom', 'tile_start', 'tile_end'] if c in df.columns]
    df = df.groupby(keep, as_index=False, sort=False).agg(raw_score=('raw_score', 'mean')).sort_values('tile_idx')

    return _bar_chart(df, df['chrom'].iloc[0], f'CRISPRi scan: {result.target_gene}' + (f' | {ontology_curie}' if ontology_curie else ''), ax=ax)

def plot_crispri_tracks(result, tile_idx, modality='rna_seq', transcript_extractor=None, zoom_width=2**15):
    if result.tile_outputs is None: 
        raise ValueError('tile_outputs is None (not available from CSV). Track plotting requires original tile_outputs.')
    tile, variant, voutput = result.tile_outputs[tile_idx]
    ctx_iv = genome.Interval(result.context_chrom, result.context_start, result.context_end)
    ref_tdata, alt_tdata = getattr(voutput.reference, modality), getattr(voutput.alternate, modality)
    components = []

    if transcript_extractor is not None:
        components.append(plot_components.TranscriptAnnotation(transcript_extractor.extract(ctx_iv)))
    components.append(plot_components.OverlaidTracks(tdata={'WT': ref_tdata, 'CRISPRi': alt_tdata}, colors={'WT': 'dimgrey', 'CRISPRi': 'red'}))
    view_iv = ref_tdata.interval.resize(zoom_width) if zoom_width else ref_tdata.interval
    # Title only — variant-name annotation would overlap with the title (5 kb
    # shuffle-variants render as a wall of bases). The tile location is in title.
    title = f'CRISPRi Tile {tile_idx+1} | {tile.chromosome}:{tile.start:,}-{tile.end:,} ({modality})'
    plot_components.plot(components, interval=view_iv, title=title)
    plt.show()

In [ ]:
#@title Genome-browser plots + BigWig export { display-mode: "form" }
"""High-quality in-notebook genome-browser views (matplotlib + AlphaGenome's
plot_components.TranscriptAnnotation for proper transcript models) PLUS
BigWig export so results open in IGV / UCSC / any genome browser at full
fidelity. CoolBox was dropped — too many rendering issues.

Each test exposes:
  plot_*_browser(result, gtf, transcript_extractor, ontology_curie=None,
                 extra_tracks=None, highlight_top_n=3, show_tss=True)
      In-notebook multi-track view. Layout:
        - Gene model (full transcript structure, exons, strand arrows)
        - Per-tile LFC bar track
        - Optional user BigWig tracks via `extra_tracks=[(label, url_or_path)]`
        - Top-N strongest hits highlighted in gold
        - TSS marked with red dashed vertical line

  export_*_bigwig(result, output_path=None, ontology_curie=None)
      Write per-tile LFC as a BigWig (or BedGraph fallback). Load in IGV
      or upload to UCSC custom tracks for full-quality genome-browser view.
"""

HG38_CHROM_SIZES = {
    'chr1': 248956422, 'chr2': 242193529, 'chr3': 198295559, 'chr4': 190214555,
    'chr5': 181538259, 'chr6': 170805979, 'chr7': 159345973, 'chr8': 145138636,
    'chr9': 138394717, 'chr10': 133797422, 'chr11': 135086622, 'chr12': 133275309,
    'chr13': 114364328, 'chr14': 107043718, 'chr15': 101991189, 'chr16': 90338345,
    'chr17': 83257441,  'chr18': 80373285,  'chr19': 58617616,  'chr20': 64444167,
    'chr21': 46709983,  'chr22': 50818468,  'chrX': 156040895,  'chrY': 57227415,
    'chrM': 16569,
}


def _tile_lfc_df(result, ontology_curie=None, source_attr='tile_effects'):
    """Normalize a result's per-tile DataFrame to chrom/start/end/raw_score."""
    df = getattr(result, source_attr).copy()
    if hasattr(result, 'target_gene') and 'gene_name' in df.columns:
        df = df[df['gene_name'] == result.target_gene]
    if ontology_curie and 'ontology_curie' in df.columns:
        df = df[df['ontology_curie'] == ontology_curie]
    elif 'ontology_curie' in df.columns and len(df) > 0:
        df = df[df['ontology_curie'] == df['ontology_curie'].iloc[0]]
    df = df.groupby(['chrom', 'tile_start', 'tile_end'], as_index=False)['raw_score'].mean()
    return df.sort_values('tile_start').reset_index(drop=True)


def _gene_tss(gtf_df, gene_symbol):
    if 'gene_name' not in gtf_df.columns: return None
    rows = gtf_df[gtf_df['gene_name'] == gene_symbol]
    if 'Feature' in rows.columns:
        gene_rows = rows[rows['Feature'] == 'gene']
        if len(gene_rows) > 0: rows = gene_rows
    if len(rows) == 0: return None
    r = rows.iloc[0]
    strand = str(r.get('Strand', '+'))
    return (str(r['Chromosome']), int(r['Start']) if strand == '+' else int(r['End']))


def _plot_browser_view(df, *, gtf, transcript_extractor, target_gene,
                       track_label, track_color, ontology_curie=None,
                       highlight_top_n=3, show_tss=True, extra_tracks=None):
    if len(df) == 0:
        print(f'No tiles to plot for {target_gene}'); return None
    chrom = df['chrom'].iloc[0]
    start, end = int(df['tile_start'].min()), int(df['tile_end'].max())
    pad = max(int((end - start) * 0.1), 1000)
    view_iv = genome.Interval(chrom, max(0, start - pad), end + pad)

    extras = extra_tracks or []
    n_axes = 2 + len(extras)
    heights = [0.8, 1.6] + [1.0] * len(extras)
    fig, axes = plt.subplots(n_axes, 1, figsize=(14, sum(heights)),
                             height_ratios=heights, sharex=True, dpi=140)
    if n_axes == 1: axes = [axes]

    title = f'{track_label}: {target_gene} | {chrom}:{start:,}-{end:,}'
    if ontology_curie: title += f' | {ontology_curie}'
    fig.suptitle(title, fontsize=12, y=0.995, fontweight='bold')

    # --- Gene track ---
    ax_gene = axes[0]
    if transcript_extractor is not None:
        try:
            transcripts = transcript_extractor.extract(view_iv)
            plot_components.TranscriptAnnotation(transcripts).plot_ax(ax_gene, axis_index=0, interval=view_iv)
        except Exception as e:
            ax_gene.text(0.5, 0.5, f'gene track unavailable: {e!s}',
                         transform=ax_gene.transAxes, ha='center', fontsize=9)
    ax_gene.set_ylabel('Genes', fontsize=9)
    for sp in ['top', 'right', 'left', 'bottom']: ax_gene.spines[sp].set_visible(False)
    ax_gene.tick_params(bottom=False, labelbottom=False, left=False, labelleft=False)

    # --- LFC bar track ---
    ax_lfc = axes[1]
    centers = ((df['tile_start'] + df['tile_end']) / 2).values
    widths = (df['tile_end'] - df['tile_start']).values * 0.92
    scores = df['raw_score'].values
    bar_colors = ['#2166ac' if s < 0 else '#b2182b' for s in scores]
    ax_lfc.bar(centers, scores, width=widths, color=bar_colors,
               edgecolor='white', linewidth=0.4)
    ax_lfc.axhline(0, color='black', lw=0.7)
    ax_lfc.set_ylabel(f'{track_label} LFC', fontsize=10)
    ax_lfc.spines[['top', 'right']].set_visible(False)
    ymax = float(np.abs(scores).max()) * 1.15 if len(scores) else 0.1
    lo = -ymax if scores.min() < 0 else 0
    hi = ymax if scores.max() > 0 else 0
    ax_lfc.set_ylim(lo or -0.01, hi or 0.01)

    # --- Highlight top hits ---
    if highlight_top_n and len(df) > 0:
        top = df.iloc[df['raw_score'].abs().sort_values(ascending=False).index].head(highlight_top_n)
        for _, r in top.iterrows():
            for ax in axes:
                ax.axvspan(r['tile_start'], r['tile_end'], color='gold', alpha=0.18, zorder=0)

    # --- TSS line ---
    if show_tss:
        tss = _gene_tss(gtf, target_gene)
        if tss and tss[0] == chrom:
            for ax in axes:
                ax.axvline(tss[1], color='red', linestyle='--', alpha=0.7, lw=1.3, zorder=1)
            axes[0].text(tss[1], 0.95, 'TSS', color='red', fontsize=8,
                         ha='center', va='top',
                         transform=axes[0].get_xaxis_transform())

    # --- Extra BigWig tracks ---
    for ax_extra, (label, path_or_url) in zip(axes[2:], extras):
        ax_extra.set_ylabel(label, fontsize=9)
        ax_extra.spines[['top', 'right']].set_visible(False)
        try:
            if not _HAS_BIGWIG:
                raise RuntimeError('pyBigWig not available')
            if not str(path_or_url).lower().endswith(('.bw', '.bigwig')):
                raise ValueError('Only .bw/.bigWig supported in extra_tracks')
            bw = pyBigWig.open(str(path_or_url))
            n_bins = 600
            vals = bw.stats(chrom, view_iv.start, view_iv.end, nBins=n_bins, type='mean')
            vals = np.array([0.0 if v is None else float(v) for v in vals])
            xs = np.linspace(view_iv.start, view_iv.end, n_bins)
            ax_extra.fill_between(xs, 0, vals, color='#4a7ba6', alpha=0.85)
            bw.close()
        except Exception as e:
            ax_extra.text(0.5, 0.5, f'(track unavailable: {e!s})',
                          transform=ax_extra.transAxes, ha='center', fontsize=8, color='gray')

    # --- Bottom x-axis ---
    axes[-1].set_xlim(view_iv.start, view_iv.end)
    axes[-1].set_xlabel(f'{chrom} (Mb)', fontsize=10)
    axes[-1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.3f}'))

    plt.tight_layout(rect=[0, 0, 1, 0.985])
    plt.show()
    return fig


def plot_necessity_browser(result, gtf, transcript_extractor=None, ontology_curie=None,
                           extra_tracks=None, highlight_top_n=3, show_tss=True):
    df = _tile_lfc_df(result, ontology_curie=ontology_curie, source_attr='tile_effects')
    return _plot_browser_view(df, gtf=gtf, transcript_extractor=transcript_extractor,
                              target_gene=result.target_gene, track_label='Necessity',
                              track_color='#2166ac', ontology_curie=ontology_curie,
                              highlight_top_n=highlight_top_n, show_tss=show_tss,
                              extra_tracks=extra_tracks)


def plot_sufficiency_browser(result, gtf, transcript_extractor=None, ontology_curie=None,
                             extra_tracks=None, highlight_top_n=3, show_tss=True):
    df = _tile_lfc_df(result, ontology_curie=ontology_curie, source_attr='tile_effects')
    return _plot_browser_view(df, gtf=gtf, transcript_extractor=transcript_extractor,
                              target_gene=result.target_gene, track_label='Sufficiency',
                              track_color='#b2182b', ontology_curie=ontology_curie,
                              highlight_top_n=highlight_top_n, show_tss=show_tss,
                              extra_tracks=extra_tracks)


def plot_crispri_browser(result, gtf, transcript_extractor=None,
                         extra_tracks=None, highlight_top_n=3, show_tss=True):
    df = _tile_lfc_df(result, source_attr='tile_scores')
    return _plot_browser_view(df, gtf=gtf, transcript_extractor=transcript_extractor,
                              target_gene=result.target_gene, track_label='CRISPRi',
                              track_color='#762a83', highlight_top_n=highlight_top_n,
                              show_tss=show_tss, extra_tracks=extra_tracks)


# --- BigWig export ----------------------------------------------------------

def _write_bigwig(df, output_path, chrom_sizes=None):
    """Write a BigWig (or BedGraph fallback on failure)."""
    df = df.sort_values(['chrom', 'tile_start']).copy()
    df['tile_start'] = df['tile_start'].astype(int)
    df['tile_end'] = df['tile_end'].astype(int)
    if _HAS_BIGWIG and output_path.endswith('.bw'):
        try:
            sizes_dict = chrom_sizes or HG38_CHROM_SIZES
            present = [c for c in df['chrom'].unique() if c in sizes_dict]
            df_bw = df[df['chrom'].isin(present)]
            bw = pyBigWig.open(output_path, 'w')
            bw.addHeader([(c, sizes_dict[c]) for c in sorted(present)])
            bw.addEntries(df_bw['chrom'].tolist(), df_bw['tile_start'].tolist(),
                          ends=df_bw['tile_end'].tolist(),
                          values=df_bw['raw_score'].astype(float).tolist())
            bw.close()
            return output_path
        except Exception as e:
            print(f'  BigWig write failed ({e!s}); writing BedGraph instead.')
    bg_path = output_path.replace('.bw', '.bedGraph')
    df[['chrom', 'tile_start', 'tile_end', 'raw_score']].to_csv(
        bg_path, sep='\t', header=False, index=False)
    return bg_path


def _lfc_to_rgb(score, max_abs):
    """Map signed LFC to a hex RGB string (BED itemRgb), red for positive, blue for negative."""
    if max_abs == 0: return '128,128,128'
    intensity = int(min(255, 80 + 175 * (abs(score) / max_abs)))
    if score < 0:
        return f'{255 - intensity},{255 - intensity},{intensity}'  # blueish
    elif score > 0:
        return f'{intensity},{255 - intensity},{255 - intensity}'  # reddish
    return '200,200,200'


def _write_bed9(df, output_path, track_name, track_description):
    """Write a BED9 file with score column + colored itemRgb so IGV/UCSC renders
    each tile as a colored box (red=positive LFC, blue=negative). Includes a
    UCSC `track` header so it auto-loads as a custom-track with proper name."""
    df = df.sort_values(['chrom', 'tile_start']).copy()
    if len(df) == 0:
        Path(output_path).write_text('')
        return output_path
    max_abs = float(df['raw_score'].abs().max()) or 1.0
    with open(output_path, 'w') as f:
        f.write(f'track name="{track_name}" description="{track_description}" itemRgb="On" visibility=2\n')
        for _, r in df.iterrows():
            score_scaled = int(round(min(1000, abs(r['raw_score']) / max_abs * 1000)))
            rgb = _lfc_to_rgb(float(r['raw_score']), max_abs)
            name = f"LFC={r['raw_score']:.3f}"
            f.write(f"{r['chrom']}\t{int(r['tile_start'])}\t{int(r['tile_end'])}\t{name}\t{score_scaled}\t.\t{int(r['tile_start'])}\t{int(r['tile_end'])}\t{rgb}\n")
    return output_path


def _export_with_summary(df, gene, test_name, color_hint=''):
    """Write both .bw and .bed; return a summary dict with paths and region info."""
    bw_path = _write_bigwig(df, f'{gene}_{test_name}.bw')
    bed_path = _write_bed9(df, f'{gene}_{test_name}.bed',
                           track_name=f'{gene}_{test_name}',
                           track_description=f'{gene} {test_name} LFC (AlphaGenome CRE)')
    chrom = df['chrom'].iloc[0] if len(df) else ''
    start = int(df['tile_start'].min()) if len(df) else 0
    end = int(df['tile_end'].max()) if len(df) else 0
    pad = max(int((end - start) * 0.2), 2_000)
    nav = f'{chrom}:{max(0, start - pad):,}-{end + pad:,}'
    summary = {
        'bigwig': bw_path,
        'bed': bed_path,
        'region': nav,
        'tile_count': len(df),
        'score_min': float(df['raw_score'].min()) if len(df) else None,
        'score_max': float(df['raw_score'].max()) if len(df) else None,
    }
    print(f'  Wrote: {bw_path}')
    print(f'  Wrote: {bed_path}')
    print()
    print(f'  In IGV / UCSC / TRAX, paste this region into the search box:')
    print(f'      {nav}')
    print(f'  ({summary["tile_count"]} tiles, score range {summary["score_min"]:.3f} … {summary["score_max"]:.3f})')
    print()
    print(f'  Tip: the BED file is usually clearer than the BigWig for sparse-tile data —')
    print(f'  it draws colored boxes (red=positive LFC, blue=negative) at each tile.')
    print(f'  In IGV, right-click the BigWig track → "Autoscale" if it looks empty.')
    return summary


def export_necessity_bigwig(result, output_path=None, ontology_curie=None):
    df = _tile_lfc_df(result, ontology_curie=ontology_curie, source_attr='tile_effects')
    return _export_with_summary(df, result.target_gene, 'necessity')


def export_sufficiency_bigwig(result, output_path=None, ontology_curie=None):
    df = _tile_lfc_df(result, ontology_curie=ontology_curie, source_attr='tile_effects')
    return _export_with_summary(df, result.target_gene, 'sufficiency')


def export_crispri_bigwig(result, output_path=None):
    df = _tile_lfc_df(result, source_attr='tile_scores')
    return _export_with_summary(df, result.target_gene, 'crispri')


In [ ]:
#@title Animation helpers (concept + real-data playback) { display-mode: "form" }
"""Two flavours:

  - concept_animation_<test>()        — mock-data schematic shown in each test's
                                        intro markdown so users can SEE what the
                                        test does before running it.
  - realdata_animation_<test>(result) — frame-by-frame playback of the actual
                                        result, revealing one tile at a time on
                                        a genome-browser-style axis.

All animations use matplotlib.animation.FuncAnimation rendered to inline JS
HTML via `.to_jshtml()`. Display inside Jupyter with `display(anim)`.
"""


def _to_anim_html(fig, update_fn, n_frames, interval_ms=600):
    anim = FuncAnimation(fig, update_fn, frames=n_frames, interval=interval_ms, blit=False, repeat=True)
    return HTML(anim.to_jshtml())


def _draw_gene_arrow(ax, name='Gene', x_start=0.55, x_end=0.7, y=0.85, color='#222'):
    ax.plot([x_start, x_end], [y, y], color=color, lw=2.5, solid_capstyle='butt')
    ax.annotate('', xy=(x_end + 0.025, y), xytext=(x_end, y),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text((x_start + x_end) / 2, y + 0.06, name, ha='center', fontsize=10, fontstyle='italic', fontweight='bold')


def _set_browser_axes(ax_g, ax_s, gene_name='TARGET'):
    ax_g.set_xlim(0, 1); ax_g.set_ylim(0, 1)
    ax_g.set_xticks([]); ax_g.set_yticks([])
    for sp in ax_g.spines.values(): sp.set_visible(False)
    _draw_gene_arrow(ax_g, name=gene_name)


def concept_animation_necessity():
    """Tile-by-tile shuffling — bars build up as critical CREs are silenced."""
    fig, (ax_g, ax_s) = plt.subplots(2, 1, figsize=(10, 4), height_ratios=[1, 1.4])
    plt.close(fig)
    n_tiles = 8
    centers = np.linspace(0.06, 0.94, n_tiles); tile_w = 0.08
    scores = np.array([0.0, -0.1, -1.8, -0.2, 0.1, -2.4, -0.3, 0.05])

    def update(frame):
        ax_g.clear(); ax_s.clear()
        _set_browser_axes(ax_g, ax_s)
        for i, cx in enumerate(centers):
            color = '#ff7f00' if i == frame else ('#cccccc' if i > frame else '#444444')
            ax_g.add_patch(plt.Rectangle((cx - tile_w/2, 0.32), tile_w, 0.22, facecolor=color, edgecolor='black', lw=0.5))
            ax_g.text(cx, 0.43, f'{i+1}', ha='center', va='center', fontsize=8, color='white' if color != '#cccccc' else 'black')
        ax_g.set_title(f'Necessity: shuffling tile {frame+1}/{n_tiles}', fontsize=11)
        ax_s.set_xlim(0, 1); ax_s.set_ylim(-3, 0.5)
        for sp in ['top', 'right']: ax_s.spines[sp].set_visible(False)
        ax_s.axhline(0, color='black', lw=0.5)
        for i in range(frame + 1):
            ax_s.bar(centers[i], scores[i], width=tile_w, color='#2166ac' if scores[i] < 0 else '#b2182b', edgecolor='white')
        ax_s.set_xticks(centers); ax_s.set_xticklabels([f'{i+1}' for i in range(n_tiles)])
        ax_s.set_xlabel('Tile'); ax_s.set_ylabel('LFC (ALT/REF)')

    return _to_anim_html(fig, update, n_frames=n_tiles, interval_ms=700)


def concept_animation_sufficiency():
    """Whole context shuffled, tile-by-tile the real CRE is planted back in.
    Positive bars = CRE was sufficient to drive expression on its own."""
    fig, (ax_g, ax_s) = plt.subplots(2, 1, figsize=(10, 4), height_ratios=[1, 1.4])
    plt.close(fig)
    n_tiles = 8
    centers = np.linspace(0.06, 0.94, n_tiles); tile_w = 0.08
    scores = np.array([0.0, 0.2, 1.5, 0.1, -0.05, 2.1, 0.3, 0.05])

    def update(frame):
        ax_g.clear(); ax_s.clear()
        _set_browser_axes(ax_g, ax_s)
        # Shuffled background across the whole strip
        ax_g.add_patch(plt.Rectangle((0, 0.30), 1, 0.26, facecolor='none', edgecolor='gray', lw=0.5, hatch='///', alpha=0.4))
        for i, cx in enumerate(centers):
            is_active = (i == frame)
            is_done = (i < frame)
            color = '#33a02c' if is_active else ('#1f7a1f' if is_done else 'none')
            alpha = 1.0 if (is_active or is_done) else 0.0
            ax_g.add_patch(plt.Rectangle((cx - tile_w/2, 0.32), tile_w, 0.22, facecolor=color, edgecolor='black', lw=0.5, alpha=alpha))
            ax_g.text(cx, 0.43, f'{i+1}', ha='center', va='center', fontsize=8, color='white' if (is_active or is_done) else 'black')
        ax_g.set_title(f'Sufficiency: planting tile {frame+1} into shuffled context', fontsize=11)
        ax_s.set_xlim(0, 1); ax_s.set_ylim(-0.5, 3)
        for sp in ['top', 'right']: ax_s.spines[sp].set_visible(False)
        ax_s.axhline(0, color='black', lw=0.5)
        for i in range(frame + 1):
            ax_s.bar(centers[i], scores[i], width=tile_w, color='#b2182b' if scores[i] > 0 else '#2166ac', edgecolor='white')
        ax_s.set_xticks(centers); ax_s.set_xticklabels([f'{i+1}' for i in range(n_tiles)])
        ax_s.set_xlabel('Tile'); ax_s.set_ylabel('LFC vs silent background')

    return _to_anim_html(fig, update, n_frames=n_tiles, interval_ms=700)


def concept_animation_crispri():
    """Fine-resolution sweep; RNA-seq pseudo-track collapses when the critical
    micro-CRE is shuffled."""
    fig, (ax_g, ax_t, ax_s) = plt.subplots(3, 1, figsize=(10, 5.2), height_ratios=[0.8, 1, 1.2])
    plt.close(fig)
    n_tiles = 12
    centers = np.linspace(0.1, 0.9, n_tiles); tile_w = 0.055
    scores = np.array([0, -0.05, -0.1, -0.5, -2.5, -0.4, -0.1, -0.05, 0, 0.05, 0, 0])
    crit = 4
    x = np.linspace(0, 1, 500)
    wt = 0.6 * np.exp(-((x - 0.65) ** 2) / 0.02) + 0.05

    def update(frame):
        for ax in (ax_g, ax_t, ax_s): ax.clear()
        for ax in (ax_g, ax_t):
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(False)
        _draw_gene_arrow(ax_g, name='TARGET', x_start=0.6, x_end=0.72, y=0.6)
        for i, cx in enumerate(centers):
            color = '#ff7f00' if i == frame else ('#cccccc' if i > frame else '#444444')
            ax_g.add_patch(plt.Rectangle((cx - tile_w/2, 0.15), tile_w, 0.2, facecolor=color, edgecolor='black', lw=0.5))
        ax_g.set_title(f'CRISPRi: shuffling fine tile {frame+1}/{n_tiles}', fontsize=11)
        # Track: WT in grey behind, ALT in red shrinking when crit tile hit
        ax_t.fill_between(x, 0.15, 0.15 + wt, color='dimgrey', alpha=0.5, label='WT')
        alt = wt.copy()
        if frame == crit: alt *= 0.18
        elif frame > crit and scores[frame] < -0.3: alt *= 0.7
        ax_t.fill_between(x, 0.15, 0.15 + alt, color='red', alpha=0.55, label='CRISPRi')
        ax_t.text(0.02, 0.78, 'RNA-seq', fontsize=8, fontweight='bold')
        ax_t.legend(loc='upper right', fontsize=7, framealpha=0.9)
        ax_s.set_xlim(0, 1); ax_s.set_ylim(-3, 0.5)
        for sp in ['top', 'right']: ax_s.spines[sp].set_visible(False)
        ax_s.axhline(0, color='black', lw=0.5)
        for i in range(frame + 1):
            ax_s.bar(centers[i], scores[i], width=tile_w, color='#2166ac' if scores[i] < 0 else '#b2182b', edgecolor='white')
        ax_s.set_xticks(centers); ax_s.set_xticklabels([f'{i+1}' for i in range(n_tiles)], fontsize=7)
        ax_s.set_xlabel('CRISPRi tile'); ax_s.set_ylabel('LFC')

    return _to_anim_html(fig, update, n_frames=n_tiles, interval_ms=600)


# --- Real-data playbacks (reveal tile-by-tile from an actual result) ---

def _playback_bar_axes(scores):
    fig, ax = plt.subplots(figsize=(12, 3.5))
    plt.close(fig)
    return fig, ax


def realdata_animation_necessity(result, ontology_curie=None):
    df = result.tile_effects.copy()
    if ontology_curie is not None and 'ontology_curie' in df.columns:
        df = df[df['ontology_curie'] == ontology_curie]
    df = df.groupby(['tile_idx', 'chrom', 'tile_start', 'tile_end'], as_index=False)['raw_score'].mean().sort_values('tile_idx')
    chrom = df['chrom'].iloc[0]
    start, end = int(df['tile_start'].min()), int(df['tile_end'].max())
    centers = ((df['tile_start'] + df['tile_end']) / 2).values
    widths = (df['tile_end'] - df['tile_start']).values * 0.9
    scores = df['raw_score'].values
    vmin = min(scores.min(), 0) * 1.1; vmax = max(scores.max(), 0) * 1.1 or 0.1
    fig, ax = _playback_bar_axes(scores)

    def update(frame):
        ax.clear()
        ax.set_xlim(start, end); ax.set_ylim(vmin, vmax)
        ax.axhline(0, color='black', lw=0.5)
        for i in range(frame + 1):
            ax.bar(centers[i], scores[i], width=widths[i], color='#2166ac' if scores[i] < 0 else '#b2182b', edgecolor='white')
        ax.set_xlabel(f'{chrom} position'); ax.set_ylabel('Necessity LFC')
        ax.set_title(f'Necessity playback: tile {frame+1}/{len(scores)} — {result.target_gene}')
        for sp in ['top', 'right']: ax.spines[sp].set_visible(False)

    return _to_anim_html(fig, update, n_frames=len(scores), interval_ms=400)


def realdata_animation_sufficiency(result, ontology_curie=None):
    df = result.tile_effects.copy()
    if ontology_curie is not None and 'ontology_curie' in df.columns:
        df = df[df['ontology_curie'] == ontology_curie]
    df = df.groupby(['tile_idx', 'chrom', 'tile_start', 'tile_end'], as_index=False)['raw_score'].mean().sort_values('tile_idx')
    chrom = df['chrom'].iloc[0]
    start, end = int(df['tile_start'].min()), int(df['tile_end'].max())
    centers = ((df['tile_start'] + df['tile_end']) / 2).values
    widths = (df['tile_end'] - df['tile_start']).values * 0.9
    scores = df['raw_score'].values
    vmin = min(scores.min(), 0) * 1.1; vmax = max(scores.max(), 0) * 1.1 or 0.1
    fig, ax = _playback_bar_axes(scores)

    def update(frame):
        ax.clear()
        ax.set_xlim(start, end); ax.set_ylim(vmin, vmax)
        ax.axhline(0, color='black', lw=0.5)
        for i in range(frame + 1):
            ax.bar(centers[i], scores[i], width=widths[i], color='#b2182b' if scores[i] > 0 else '#2166ac', edgecolor='white')
        ax.set_xlabel(f'{chrom} position'); ax.set_ylabel('Sufficiency LFC')
        ax.set_title(f'Sufficiency playback: tile {frame+1}/{len(scores)} — {result.target_gene}')
        for sp in ['top', 'right']: ax.spines[sp].set_visible(False)

    return _to_anim_html(fig, update, n_frames=len(scores), interval_ms=400)


def realdata_animation_crispri(result):
    df = result.tile_scores[result.tile_scores['gene_name'] == result.target_gene].copy()
    df = df.groupby(['tile_idx', 'chrom', 'tile_start', 'tile_end'], as_index=False)['raw_score'].mean().sort_values('tile_idx')
    chrom = df['chrom'].iloc[0]
    start, end = int(df['tile_start'].min()), int(df['tile_end'].max())
    centers = ((df['tile_start'] + df['tile_end']) / 2).values
    widths = (df['tile_end'] - df['tile_start']).values * 0.9
    scores = df['raw_score'].values
    vmin = min(scores.min(), 0) * 1.1; vmax = max(scores.max(), 0) * 1.1 or 0.1
    fig, ax = _playback_bar_axes(scores)

    def update(frame):
        ax.clear()
        ax.set_xlim(start, end); ax.set_ylim(vmin, vmax)
        ax.axhline(0, color='black', lw=0.5)
        for i in range(frame + 1):
            ax.bar(centers[i], scores[i], width=widths[i], color='#2166ac' if scores[i] < 0 else '#b2182b', edgecolor='white')
        ax.set_xlabel(f'{chrom} position'); ax.set_ylabel('CRISPRi LFC')
        ax.set_title(f'CRISPRi playback: tile {frame+1}/{len(scores)} — {result.target_gene}')
        for sp in ['top', 'right']: ax.spines[sp].set_visible(False)

    return _to_anim_html(fig, update, n_frames=len(scores), interval_ms=400)


In [ ]:
#@title Connect to AlphaGenome { display-mode: "form" }
dna_model, gtf, transcript_extractor = init(api_key=userdata.get('ALPHAGENOME_API_KEY'))

In [ ]:
#@title Search cell types (optional helper) { display-mode: "form" }
search_query = 'CD34'  #@param {type:"string"}
search_cell_types(dna_model, search_query)

In [ ]:
#@title 1. Experiment setup { display-mode: "form" }
#@markdown **Gene** — symbol of the gene whose regulation you want to dissect (GENCODE v46 / hg38).
gene = 'TAL1'  #@param {type:"string"}
#@markdown **Cell type** — ontology CURIE identifying the cell type / tissue / line. Use the **Search cell types** helper above to find it.
ontology_curie = 'CL:0001059'  #@param {type:"string"}
#@markdown ---
#@markdown ### Shuffle replicates
#@markdown **Used by:** Necessity, Higher-Order Interaction, and CRISPRi.
#@markdown **NOT used by:** Sufficiency — that test has its own `sufficiency_n_shuffles` setting (the shuffle there is over the whole 1 MB context, so it's much more expensive per replicate).
#@markdown
#@markdown **What is a *replicate*?** One independent dinucleotide-shuffled version of a tile's sequence. The model is scored on this perturbed sequence; results across N replicates are averaged to get a stable per-tile score (so you're not at the mercy of one unlucky random permutation).
#@markdown
#@markdown **What is a *dinucleotide shuffle*?** A randomization of the DNA letters that **preserves the frequency of every adjacent letter-pair** (AA, AC, AG, AT, CA, ...). This destroys transcription-factor binding sites and regulatory grammar while keeping GC content and local sequence composition identical — a strong null where nothing biologically meaningful is left in the tile.
#@markdown
#@markdown **Cost.** Each replicate is scored in both forward and reverse-complement orientations, so the per-tile API cost is `4 × replicates`. Higher = less noise, more runtime.
shuffle_replicates = 2  #@param {type:"integer"}

gene = gene.strip()
ontology_curie = ontology_curie.strip()

myexp = CREExperiment(
    model=dna_model, gene_symbol=gene, gtf=gtf,
    transcript_extractor=transcript_extractor, ontology_terms=[ontology_curie],
)

gene_iv = gene_annotation.get_gene_interval(gtf, gene_symbol=gene)
print(f'Gene: {gene}')
print(f'Gene interval: {gene_iv}')
print(f'Ontology: {ontology_curie}')
print(f'Shuffle replicates: {shuffle_replicates}')


In [ ]:
#@title 2. Locus tiling — for Necessity + Interaction { display-mode: "form" }
#@markdown These settings only affect the **Necessity** and **Higher-Order Interaction** tests, which tile a window centred on the gene. If you only plan to run the CRISPRi scan, you can skip this cell.
#@markdown
#@markdown **Scan region width (bp)** — genomic window centred on the gene. 40 kb covers proximal enhancers; 100–200 kb captures distal CREs.
scan_region_width_bp = 40000  #@param {type:"integer"}
#@markdown **Tile width (bp)** — coarse tile size. 5 kb is the enhancer-scale default.
tile_width_bp = 5000  #@param {type:"integer"}

tile_region = gene_iv.resize(scan_region_width_bp)
tiles = CREExperiment.tile_region(tile_region, tile_width=tile_width_bp)
print(f'Coarse tiles: {len(tiles)} x {tile_width_bp} bp covering {tile_region}')


## Necessity Test

**What this measures.** Which CREs are **required** for your gene's expression in this cell type. We tile the locus, computationally "silence" each tile by replacing it with a sequence-composition-matched shuffle, and measure the drop in predicted gene expression. Tiles with strongly negative bars are essential CREs (enhancers/promoter); positive bars indicate silencers.

**Concept** ([CREME](https://www.nature.com/articles/s41588-024-01923-3), Toneyan and Koo, Nature Genetics 2024): Tile a locus into fixed-width windows, replace each tile with a dinucleotide-preserving shuffle, and measure the effect on target gene expression. Tiles whose perturbation strongly reduces expression are **necessary** CREs.

In [ ]:
#@title 3. Run necessity test { display-mode: "form" }
necessity = myexp.necessity_test(tiles=tiles, target_gene=gene, n_shuffles=shuffle_replicates)
print(f'Total score rows: {len(necessity.scores)}')
print(f'Tile effects for {gene}: {len(necessity.tile_effects)} rows')
necessity.tile_effects[['tile_label', 'ontology_curie', 'raw_score', 'quantile_score']].head(20)


In [ ]:
#@title Necessity — bar chart { display-mode: "form" }
if 'necessity' not in globals():
    print('Run the Necessity test cell above first.')
else:
    plot_necessity_bar(necessity, ontology_curie=ontology_curie)
    plt.show()


In [ ]:
#@title Necessity — genome browser view { display-mode: "form" }
# To overlay user BigWigs (e.g. ENCODE DNase/H3K27ac), pass extra_tracks=[(label, url), ...].
if 'necessity' not in globals():
    print('Run the Necessity test cell above first.')
else:
    plot_necessity_browser(necessity, gtf=gtf, transcript_extractor=transcript_extractor,
                           ontology_curie=ontology_curie)


In [ ]:
#@title Necessity — export BigWig + BED (open in IGV/UCSC/TRAX) { display-mode: "form" }
from google.colab import files as _colab_files
if 'necessity' not in globals():
    print('Run the Necessity test cell above first.')
else:
    info = export_necessity_bigwig(necessity, ontology_curie=ontology_curie)
    _colab_files.download(info['bed'])
    _colab_files.download(info['bigwig'])


## Sufficiency Test

**What this measures.** Whether each CRE is **sufficient on its own** to drive your gene's expression. We dinucleotide-shuffle the *entire* 1 MB context (effectively erasing all regulatory information), then plant one tile at a time back into the shuffled background and ask the model what expression looks like. Tiles with strongly positive bars are CREs that can act alone — typically the promoter and dominant enhancers.

**Concept.** This is the inverse of the necessity test:
- *Necessity:* "what happens if I remove **this** CRE?"
- *Sufficiency:* "what happens if I remove **everything but** this CRE?"

A CRE that is both **necessary and sufficient** is a strong solo driver. A CRE that is necessary but *not* sufficient probably acts in cooperation with other elements — interesting biology, often the kind of CRE that the higher-order interaction test will pull out.

**Cost.** Each replicate sends ~1 MB of sequence per orientation (vs. ~5 kb for necessity). Keep `sufficiency_n_shuffles` small (1–3) and the tile count moderate.

In [ ]:
#@title Sufficiency parameters { display-mode: "form" }
#@markdown **Background shuffles** — each tile is tested against this many independent fully-shuffled contexts (~1 MB per replicate per orientation). Keep small to control API cost.
sufficiency_n_shuffles = 2  #@param {type:"integer"}


In [ ]:
#@title 3b. Run sufficiency test { display-mode: "form" }
sufficiency = myexp.sufficiency_test(tiles=tiles, target_gene=gene, n_shuffles=sufficiency_n_shuffles)
print(f'Total score rows: {len(sufficiency.scores)}')
print(f'Tile effects for {gene}: {len(sufficiency.tile_effects)} rows')
sufficiency.tile_effects[['tile_label', 'ontology_curie', 'raw_score', 'quantile_score']].head(20)


In [ ]:
#@title Sufficiency — genome browser view { display-mode: "form" }
# To overlay user BigWigs, pass extra_tracks=[(label, url), ...].
if 'sufficiency' not in globals():
    print('Run the Sufficiency test cell above first.')
else:
    plot_sufficiency_browser(sufficiency, gtf=gtf, transcript_extractor=transcript_extractor,
                             ontology_curie=ontology_curie)


In [ ]:
#@title Sufficiency — export BigWig + BED (open in IGV/UCSC/TRAX) { display-mode: "form" }
from google.colab import files as _colab_files
if 'sufficiency' not in globals():
    print('Run the Sufficiency test cell above first.')
else:
    info = export_sufficiency_bigwig(sufficiency, ontology_curie=ontology_curie)
    _colab_files.download(info['bed'])
    _colab_files.download(info['bigwig'])


In [ ]:
#@title Sufficiency — playback animation { display-mode: "form" }
if 'sufficiency' not in globals():
    print('Run the Sufficiency test cell above first.')
else:
    display(realdata_animation_sufficiency(sufficiency, ontology_curie=ontology_curie))


## Higher-Order Interaction Test

**What this measures.** Which CREs **cooperate**. We identify the strongest single CRE, permanently remove it, then ask which CRE becomes the new dominant driver. Iterating this reveals redundancy and hierarchy — useful for spotting shadow enhancers and compensating elements.

**Concept**: Starting from the necessity results, run **greedy ablation** (CREME's `higher_order_interaction_test`). At each round, the tile with the largest remaining effect is permanently removed, and all other tiles are re-scored. This reveals how CRE contributions compound — e.g., whether removing the MuTE enhancer exposes dependency on a secondary element.

Under a linear-additivity approximation, each tile is scored independently per round and the cumulative LFC is accumulated. This is the same greedy strategy CREME uses, adapted for AlphaGenome's variant API.

In [ ]:
#@title Interaction parameters { display-mode: "form" }
#@markdown **Ablation rounds** — each round removes the strongest remaining tile and re-scores the rest. More rounds = deeper cooperativity revealed.
greedy_ablation_rounds = 4  #@param {type:"integer"}


In [ ]:
#@title 4. Run higher-order interaction test { display-mode: "form" }
interaction = myexp.interaction_test(
    tiles=tiles, target_gene=gene, num_rounds=greedy_ablation_rounds, n_shuffles=shuffle_replicates,
)


In [ ]:
#@title Interaction — waterfall plot { display-mode: "form" }
if 'interaction' not in globals():
    print('Run the Interaction test cell above first.')
else:
    plot_interaction_waterfall(interaction)
    plt.show()


In [ ]:
#@title Interaction — contribution plot { display-mode: "form" }
if 'interaction' not in globals():
    print('Run the Interaction test cell above first.')
else:
    plot_interaction_contribution(interaction)
    plt.show()


## CRISPRi Tiling Scan

**What this measures.** A simulated **dCas9-KRAB CRISPRi tiling screen** over a chosen region. Each fine-resolution tile is shuffled and full RNA-seq (± chromatin) tracks are returned — letting you see exactly *where* and *how* expression collapses, in the same readout format you'd get from a wet-lab CRISPRi screen.

**Concept**: A focused, fine-resolution perturbation scan of a specific regulatory region. This mirrors CRISPRi tiling screens — tile a candidate enhancer with small windows, replace each with a dinucleotide shuffle, and compare full WT vs perturbed tracks.

For each perturbation we get:
- **Scalar scores** (GeneMaskLFCScorer) for quantitative comparison (averaged over shuffle replicates)
- **Full REF/ALT tracks** (predict_variant) for visual inspection of RNA-seq changes

In [ ]:
#@title CRISPRi parameters { display-mode: "form" }
#@markdown **Target region** — `chrN:start-end`. Leave blank to auto-centre a window on the gene.
crispri_target_region = ''  #@param {type:"string"}
#@markdown **Flank around gene (kb)** — used only when target region is blank. 5 kb each side covers most proximal CREs.
crispri_flank_kb = 5  #@param {type:"integer"}
#@markdown **CRISPRi tile width (bp)** — fine-resolution tile size, mimics CRISPRi guide tiling.
crispri_tile_width_bp = 1000  #@param {type:"integer"}
#@markdown **Output tracks** — RNA-seq always; optionally add chromatin accessibility.
crispri_output_tracks = "RNA_SEQ"  #@param ["RNA_SEQ", "RNA_SEQ + DNASE", "RNA_SEQ + ATAC"]


In [ ]:
#@title 5. Run CRISPRi tiling scan { display-mode: "form" }
crispri_region_str = crispri_target_region.strip()

if crispri_region_str:
    m = re.match(r'^(chr[\w]+):([\d,]+)-([\d,]+)$', crispri_region_str)
    if not m:
        raise ValueError(f'Invalid region {crispri_region_str!r}. Expected e.g. chr1:12345-67890')
    scan_region = genome.Interval(m.group(1),
                                  int(m.group(2).replace(',', '')),
                                  int(m.group(3).replace(',', '')))
    print(f'Using explicit scan region: {scan_region}')
else:
    scan_region = gene_iv.resize(int(crispri_flank_kb * 2 * 1000))
    print(f'Auto-anchored scan region around {gene}: {scan_region}')
    if 'necessity' in globals():
        try:
            te = necessity.tile_effects
            te = te[te['ontology_curie'] == ontology_curie]
            worst = te.loc[te['raw_score'].idxmin(), 'tile_label']
            print(f'  Hint: necessity test flagged {worst} as the strongest hit — consider entering an explicit region near it.')
        except Exception:
            pass

crispri_tiles = CREExperiment.tile_region(scan_region, tile_width=crispri_tile_width_bp)
print(f'CRISPRi: {len(crispri_tiles)} x {crispri_tile_width_bp} bp tiles')

output_map = {
    'RNA_SEQ':          [dna_client.OutputType.RNA_SEQ],
    'RNA_SEQ + DNASE':  [dna_client.OutputType.RNA_SEQ, dna_client.OutputType.DNASE],
    'RNA_SEQ + ATAC':   [dna_client.OutputType.RNA_SEQ, dna_client.OutputType.ATAC],
}
crispri = myexp.crispri_scan(tiles=crispri_tiles,
                             outputs=output_map[crispri_output_tracks],
                             target_gene=gene,
                             n_shuffles=shuffle_replicates)
print(f'CRISPRi scan complete: {len(crispri.tile_outputs)} tiles scored.')


In [ ]:
#@title CRISPRi — summary bar chart { display-mode: "form" }
if 'crispri' not in globals():
    print('Run the CRISPRi scan cell above first.')
else:
    plot_crispri_summary(crispri)
    plt.show()


In [ ]:
#@title CRISPRi — genome browser view { display-mode: "form" }
# To overlay user BigWigs, pass extra_tracks=[(label, url), ...].
if 'crispri' not in globals():
    print('Run the CRISPRi scan cell above first.')
else:
    plot_crispri_browser(crispri, gtf=gtf, transcript_extractor=transcript_extractor)


In [ ]:
#@title CRISPRi — export BigWig + BED (open in IGV/UCSC/TRAX) { display-mode: "form" }
from google.colab import files as _colab_files
if 'crispri' not in globals():
    print('Run the CRISPRi scan cell above first.')
else:
    info = export_crispri_bigwig(crispri)
    _colab_files.download(info['bed'])
    _colab_files.download(info['bigwig'])


In [ ]:
#@title CRISPRi — track view (auto-selects best tile) { display-mode: "form" }
if 'crispri' not in globals():
    print('Run the CRISPRi scan cell above first.')
else:
    gene_scores = crispri.tile_scores[crispri.tile_scores['gene_name'] == gene]
    if len(gene_scores) > 0:
        best_idx = gene_scores['raw_score'].abs().idxmax(); best_label = gene_scores.loc[best_idx, 'tile_label']
        ti = int(best_label.split()[-1]) - 1
        print(f'Most impactful tile: {best_label} ({crispri_tiles[ti]})')
        print(f'LFC = {gene_scores.loc[best_idx, "raw_score"]:.4f}')
        plot_crispri_tracks(crispri, tile_idx=ti, modality='rna_seq', transcript_extractor=transcript_extractor, zoom_width=2**15)
    else:
        print(f'{gene} not found in CRISPRi scores. Check context interval.')


In [ ]:
#@title 6. Download results { display-mode: "form" }
from google.colab import files

result_dir = f'{gene}_{ontology_curie.replace(":", "_")}_results'
os.makedirs(result_dir, exist_ok=True)
saved = []
if 'necessity' in globals():
    necessity.to_csv(f'{result_dir}/necessity'); saved.append('necessity')
if 'sufficiency' in globals():
    sufficiency.to_csv(f'{result_dir}/sufficiency'); saved.append('sufficiency')
if 'interaction' in globals():
    interaction.to_csv(f'{result_dir}/interaction'); saved.append('interaction')
if 'crispri' in globals():
    crispri.to_csv(f'{result_dir}/crispri'); saved.append('crispri')

if not saved:
    print('Nothing to download — run at least one test first.')
else:
    zip_name = f'{gene}_{ontology_curie.replace(":", "_")}.results.zip'
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, filenames in os.walk(result_dir):
            for fn in filenames: zf.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), '.'))

    print(f'Saved experiments: {", ".join(saved)}')
    print(f'Downloading {zip_name}...')
    files.download(zip_name)


---
## Appendix: Quick Reference

| Task | API Method | Key Parameters |
|------|-----------|----------------|
| Predict from sequence | `dna_model.predict_sequence()` | `sequence`, `requested_outputs`, `ontology_terms` |
| Predict from interval | `dna_model.predict_interval()` | `interval`, `requested_outputs`, `ontology_terms` |
| Predict variant (REF vs ALT) | `dna_model.predict_variant()` | `interval`, `variant`, `requested_outputs`, `ontology_terms` |
| Score a variant | `dna_model.score_variant()` | `interval`, `variant`, `variant_scorers` |
| Score many variants | `dna_model.score_variants()` | `intervals`, `variants`, `variant_scorers` |
| ISM | `dna_model.score_ism_variants()` | `interval`, `ism_interval`, `variant_scorers` |
| **CREME necessity** | `CREExperiment.necessity_test()` | `tiles`, `target_gene`, `n_shuffles` |
| **CREME interaction** | `CREExperiment.interaction_test()` | `tiles`, `num_rounds`, `ontology_curie`, `n_shuffles` |
| **CREME CRISPRi scan** | `CREExperiment.crispri_scan()` | `tiles`, `outputs`, `target_gene`, `n_shuffles` |
| **Cell type search** | `search_cell_types()` | `model`, `query` |

### Supported sequence lengths
- `SEQUENCE_LENGTH_16KB` = 16,384 bp
- `SEQUENCE_LENGTH_100KB` = 131,072 bp
- `SEQUENCE_LENGTH_500KB` = 524,288 bp
- `SEQUENCE_LENGTH_1MB` = 1,048,576 bp

### Key variant scorers
- `GeneMaskLFCScorer` — Log fold change per gene (gene-centric, signed)
- `CenterMaskScorer` — Effect in a window around the variant (non-gene-centric)
- `GeneMaskSplicingScorer` — Splicing changes per gene
- `SpliceJunctionScorer` — Splice junction disruption
- `ContactMapScorer` — 3D chromatin contact disruption
- `PolyadenylationScorer` — paQTL effects

### Ontology resources
- UBERON (anatomy): https://www.ebi.ac.uk/ols4/ontologies/uberon
- Cell Ontology: https://www.ebi.ac.uk/ols4/ontologies/cl
- EFO (cell lines): https://www.ebi.ac.uk/ols4/ontologies/efo

### Documentation
- Full docs: https://www.alphagenomedocs.com/
- Variant scoring: https://www.alphagenomedocs.com/variant_scoring.html
- Visualization: https://www.alphagenomedocs.com/visualization_library_basics.html